In [1]:
import os
import csv
import pandas as pd
from datetime import datetime
from pyflink.table.expressions import col, lit
from pyflink.table.window import Slide, Tumble
from pyflink.table.udf import udf
from pyflink.table import (
    EnvironmentSettings,
    TableEnvironment,
    DataTypes
)

In [2]:
env_settings = (
    EnvironmentSettings.new_instance()
    .in_streaming_mode()
    .build()
)
t_env = TableEnvironment.create(env_settings)
conf = t_env.get_config().get_configuration()
conf.set_string("execution.target", "remote")
conf.set_string("rest.address", "jobmanager")
conf.set_string("rest.port", "8081")
conf.set_string("parallelism.default", "1")

/usr/local/lib/python3.11/dist-packages/apache_beam/runners/portability/stager.py:63: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
columns = ["agg_trade_id","price","quantity","first_trade_id","last_trade_id","timestamp","is_buyer_maker","is_best_match"]
df_pd = pd.read_csv("/workspace/ADAUSDT-aggTrades-2025-09-27.csv", header=None, names=columns, nrows=1000)

In [4]:
df_pd.dtypes

agg_trade_id        int64
price             float64
quantity          float64
first_trade_id      int64
last_trade_id       int64
timestamp           int64
is_buyer_maker       bool
is_best_match        bool
dtype: object

In [5]:
t_env.from_pandas(df_pd).print_schema()

(
  `agg_trade_id` BIGINT,
  `price` DOUBLE,
  `quantity` DOUBLE,
  `first_trade_id` BIGINT,
  `last_trade_id` BIGINT,
  `timestamp` BIGINT,
  `is_buyer_maker` BOOLEAN,
  `is_best_match` BOOLEAN
)


In [3]:
t_env.execute_sql("""
create table aggtrades_source (
  `agg_trade_id` BIGINT,
  `price` DOUBLE,
  `quantity` DOUBLE,
  `first_trade_id` BIGINT,
  `last_trade_id` BIGINT,
  `timestamp` BIGINT,
  `is_buyer_maker` BOOLEAN,
  `is_best_match` BOOLEAN
) with (
    'connector' = 'filesystem',
    'path' = '/workspace/ADAUSDT-aggTrades-2025-09-27.csv',
    'format' = 'csv'
)
""")

In [4]:
aggtrades_source = t_env.from_path("aggtrades_source")

In [5]:
aggtrades_source.print_schema()

(
  `agg_trade_id` BIGINT,
  `price` DOUBLE,
  `quantity` DOUBLE,
  `first_trade_id` BIGINT,
  `last_trade_id` BIGINT,
  `timestamp` BIGINT,
  `is_buyer_maker` BOOLEAN,
  `is_best_match` BOOLEAN
)


In [6]:
aggtrades_source.to_pandas().tail()

,agg_trade_id,price,quantity,first_trade_id,last_trade_id,timestamp,is_buyer_maker,is_best_match
38238,411153005,0.7810,324.5,711818870,711818872,1759017560455452,False,True
38239,411153006,0.7810,433.6,711818873,711818873,1759017566049126,False,True
38240,411153007,0.7810,23.9,711818874,711818874,1759017582090484,False,True
38241,411153008,0.7810,15.0,711818875,711818875,1759017598615357,False,True
38242,411153009,0.7809,30.2,711818876,711818876,1759017599501352,True,True


In [3]:
t_env.execute_sql("""
create table aggtrades_sink (
  `agg_trade_id` BIGINT,
  `price` DOUBLE,
  `quantity` DOUBLE,
  `first_trade_id` BIGINT,
  `last_trade_id` BIGINT,
  `timestamp` BIGINT,
  `is_buyer_maker` BOOLEAN,
  `is_best_match` BOOLEAN,
  `ingest_date` DATE,
  `ingest_timestamp` TIMESTAMP(3)
) with (
    'connector' = 'filesystem',
    'path' = '/workspace/output/aggTrades/ADAUSDT/2025-09-27',
    'format' = 'csv'
)
""")

In [8]:
t_env.execute_sql("""
INSERT INTO aggtrades_sink
SELECT *, CURRENT_DATE, CURRENT_TIMESTAMP from aggtrades_source
""")

In [9]:
sink_table = t_env.from_path("aggtrades_sink")

In [10]:
sink_table.print_schema()

(
  `agg_trade_id` BIGINT,
  `price` DOUBLE,
  `quantity` DOUBLE,
  `first_trade_id` BIGINT,
  `last_trade_id` BIGINT,
  `timestamp` BIGINT,
  `is_buyer_maker` BOOLEAN,
  `is_best_match` BOOLEAN,
  `ingest_date` DATE,
  `ingest_timestamp` TIMESTAMP(3)
)


In [11]:
sink_table.to_pandas().tail()

,agg_trade_id,price,quantity,first_trade_id,last_trade_id,timestamp,is_buyer_maker,is_best_match,ingest_date,ingest_timestamp
114724,411153005,0.7810,324.5,711818870,711818872,1759017560455452,False,True,2025-10-12,2025-10-12 05:36:38.307
114725,411153006,0.7810,433.6,711818873,711818873,1759017566049126,False,True,2025-10-12,2025-10-12 05:36:38.307
114726,411153007,0.7810,23.9,711818874,711818874,1759017582090484,False,True,2025-10-12,2025-10-12 05:36:38.307
114727,411153008,0.7810,15.0,711818875,711818875,1759017598615357,False,True,2025-10-12,2025-10-12 05:36:38.308
114728,411153009,0.7809,30.2,711818876,711818876,1759017599501352,True,True,2025-10-12,2025-10-12 05:36:38.308


In [12]:
sink_table.to_pandas().count()

agg_trade_id        114729
price               114729
quantity            114729
first_trade_id      114729
last_trade_id       114729
timestamp           114729
is_buyer_maker      114729
is_best_match       114729
ingest_date         114729
ingest_timestamp    114729
dtype: int64

In [ ]:
t_env.execute_sql("""
SELECT count(*) from aggtrades_source
""").print()

In [7]:
t_table = t_env.execute_sql("""
SELECT * from aggtrades_sink limit 10
""")